# 1단계 · 새 VGGFace2 Parent 학습과 검증

같은 RUN_ID를 사용합니다. [실행과 결과 안내](HOW_TO_READ.md)

In [ ]:
# cell 1 : 설정 및 경로 확인
import json
import os
import sys
from pathlib import Path

from IPython.display import display
import pandas as pd

repo = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "fine-tune/modules").is_dir())
sys.path.insert(0, str(repo / "fine-tune"))
from modules.clean_training import final_artifact, recipe, run_directory, run_stage
from modules.clean_campaign import model_dir, origin_matrix, parent_validation, read_result, run_full, run_probe, validate_choices

DATA_ROOT = Path(os.environ.get("FACE_DATASET_ROOT", "F:/face_datasets"))
MODEL_ROOT = Path(os.environ.get("FACE_MODEL_ROOT", "F:/models"))
RUN_ID = "carbon_mfn1024_s8972_v2"
SEED = 8972
EXECUTE = False
RESUME = False
KEEP_RAW_RESULTS = False
VALIDATE_PARENT_ONLY = False
run = run_directory(MODEL_ROOT, RUN_ID)
print(run / "parent")
display(recipe("parent"))

In [ ]:
# cell 2 : 실행
if EXECUTE:
    if not VALIDATE_PARENT_ONLY and not (run / "parent/COMPLETED").is_file():
        run_stage(DATA_ROOT, run, "parent", seed=SEED, keep_raw_results=KEEP_RAW_RESULTS, resume=RESUME)
    parent_result = parent_validation(DATA_ROOT, run, seed=SEED, keep_raw_results=KEEP_RAW_RESULTS)

In [ ]:
# cell 3 : 결과 확인
if EXECUTE:
    display(pd.DataFrame(parent_result["comparison"]))
elif (run / "parent_validation/COMPLETED").is_file():
    display(pd.DataFrame(read_result(run / "parent_validation")["comparison"]))
else:
    progress_path = run / "parent/progress.json"
    if progress_path.is_file():
        epochs = json.loads(progress_path.read_bytes())["epochs"]
        display(pd.DataFrame([
            dict(epoch=r["epoch"], lfw=100*r["zero_shot"]["metrics"]["lfw"]["accuracy"],
                 scface=100*r["zero_shot"]["metrics"]["scface"]["rank1"],
                 tinyface=100*r["zero_shot"]["metrics"]["tinyface"]["rank1"],
                 seconds=r["zero_shot"]["seconds"])
            for r in epochs if "zero_shot" in r
        ]))
    print("완료 후 단독 검증:", run / "parent_validation/summary.json")